# Pillar 2 — Wrong-Answer Preference Structure (Figure 4)

PathOPEN's wrong answers are pathologist-authored and graded by difficulty
(near-miss / moderate-miss / far-miss). No comparator dataset has wrong answers at all,
so this pillar is validated entirely within PathOPEN.

**Figure 4**
- **a** Tier distribution, OE-native vs MCQ-derived wrong answers
- **b** Judge-vs-pathologist tier agreement (confusion matrix), weighted κ annotated
- **c** Behavioural validation: does a near-miss distractor actually fool models more
  often than a far-miss one?

## What each panel can and cannot claim

**a is the claim.** All three tiers are populated, so a graded structure exists. This is
descriptive and it is solid.

**b is the weak point, and the figure should show that.** Tier κ is 0.02–0.25 across
judges and criteria — poor to fair. The confusion matrices explain why: judges
systematically over-rate distractor plausibility, calling far-miss options near-misses.
Reporting only the tier distribution while omitting this would overstate the validation.

**c is the paper's own "strongest evidence"** (§2.2 reviewer concerns) precisely because
b is weak — a behavioural test does not depend on a judge agreeing with a rubric. It is
marked optional in the paper; the answerer runs make it computable.

Its honest reading: 5 of 6 models show attraction decreasing from near-miss to far-miss,
the predicted direction, but with three ordered bins **Kendall's τ cannot reach
significance** (best *P* = 0.333). Directionally consistent, not confirmed. The panel is
drawn so a reader sees the spread and the sample sizes rather than a clean gradient that
is not there.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nature_style as ns

ns.apply_style()

EVAL = os.path.join("..", "data_evaluation")
AGREE = os.path.join(EVAL, "vlm", "agreement_output")
ZEROSHOT = os.path.join(EVAL, "vlm_zeroshot", "analysis")

TIER_ORDER = ["near-miss", "moderate-miss", "far-miss"]
TIER_SHORT = {"near-miss": "Near", "moderate-miss": "Moderate", "far-miss": "Far"}
# Sequential, dark = hardest. A diverging palette would imply a midpoint the tiers
# do not have.
TIER_COLORS = {"near-miss": "#1A4E8A", "moderate-miss": "#6BA3D6", "far-miss": "#C6DBEF"}

print("style applied")

## Panel a — tier distribution

Pooled across the five pathologists. Split by source because §2.2 Innovation 3 claims
coverage across both OE-native and MCQ-derived wrong answers, and the two distributions
differ materially.

In [ ]:
import ast

tiers = pd.read_csv(os.path.join(AGREE, "wrong_answer_tier_agreement.csv"))
tiers["counts"] = tiers.tier_counts.apply(
    lambda v: ast.literal_eval(v) if isinstance(v, str) else v)

# tier_counts are the HUMAN tiers, identical across judges - take one judge to avoid
# counting every item twice.
one_judge = tiers[tiers.judge == "qwenvl"]

distribution = {}
for source in ("OE_native", "MCQ_derived"):
    subset = one_judge[one_judge.source == source]
    pooled = {2: 0, 1: 0, 0: 0}
    for counts in subset.counts:
        for score, count in counts.items():
            pooled[int(score)] += count
    total = sum(pooled.values())
    distribution[source] = {
        "near-miss": pooled[2], "moderate-miss": pooled[1], "far-miss": pooled[0],
        "total": total,
    }
    print(f"{source:12s} near {pooled[2]:4d} ({100*pooled[2]/total:4.1f}%)  "
          f"moderate {pooled[1]:4d} ({100*pooled[1]/total:4.1f}%)  "
          f"far {pooled[0]:4d} ({100*pooled[0]/total:4.1f}%)   n={total}")

## Panel b — tier agreement confusion matrices

Recomputed from the pooled ratings, because `wrong_answer_tier_agreement.csv` stores κ
and tier counts but not the matrices themselves.

Rows are the pathologist's tier, columns the judge's. A diagonal-heavy matrix means the
judge reproduces the tiering; mass **below** the diagonal means the judge rates
distractors as harder (more near-miss) than the pathologist did.

In [ ]:
import glob

HUMAN_INPUT = os.path.join(EVAL, "pathologists", "scoring_analysis", "input")
JUDGE_OUTPUT = os.path.join(EVAL, "vlm", "judge_output")

# Same column map and binning as ../data_evaluation/vlm/wrong_answer_tier_agreement.ipynb:
# score 2 -> near, 1 -> moderate, 0 -> far, -1 -> excluded (no tier).
WRONG_COLUMNS = []
for i in (1, 2):
    WRONG_COLUMNS.append(("OE_native",
                          f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)",
                          f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)"))
for i in (1, 2, 3, 4):
    WRONG_COLUMNS.append(("MCQ_derived",
                          f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)",
                          f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)"))


def pooled_human():
    frames = []
    for directory in sorted(glob.glob(os.path.join(HUMAN_INPUT, "evaluator[0-9]*")),
                            key=lambda p: int(os.path.basename(p).replace("evaluator", ""))):
        path = os.path.join(directory, "pathopen_eval_data.csv")
        if os.path.exists(path):
            frames.append(pd.read_csv(path).drop(index=0).reset_index(drop=True))
    return pd.concat(frames, ignore_index=True)


def confusion(judge_key):
    """3x3 counts, rows = human tier, cols = judge tier. Both raters must have a tier."""
    human = pooled_human()
    judge = pd.read_csv(os.path.join(JUDGE_OUTPUT, f"evaluator_{judge_key}",
                                     "pathopen_eval_data.csv"))
    merged = human.merge(judge, on="Image_ID", suffixes=("_human", "_judge"))
    matrix = np.zeros((3, 3), dtype=int)
    index_of = {2: 0, 1: 1, 0: 2}          # near, moderate, far
    for _, human_col, judge_col in WRONG_COLUMNS:
        h = pd.to_numeric(merged.get(f"{human_col}_human", merged.get(human_col)),
                          errors="coerce")
        j = pd.to_numeric(merged.get(f"{judge_col}_judge", merged.get(judge_col)),
                          errors="coerce")
        for hv, jv in zip(h, j):
            if hv in index_of and jv in index_of:
                matrix[index_of[hv], index_of[jv]] += 1
    return matrix


matrices = {k: confusion(k) for k in ("internvl", "qwenvl")}
for key, matrix in matrices.items():
    on_diagonal = np.trace(matrix) / matrix.sum()
    below = np.tril(matrix, -1).sum() / matrix.sum()   # judge rated HARDER than human
    print(f"{key}: n={matrix.sum()}  on-diagonal {100*on_diagonal:.1f}%  "
          f"judge-rates-harder {100*below:.1f}%")
    print(matrix)

## Panel c — behavioural validation

From `tier_difficulty.py`. Attraction rate is the percentage of *offered* distractors of
each tier that the model actually selected — normalised, so a tier that simply appears
more often does not look more attractive.

In [ ]:
attraction = pd.read_csv(os.path.join(ZEROSHOT, "tier_distractor_attraction.csv"))
monotonicity = pd.read_csv(os.path.join(ZEROSHOT, "tier_monotonicity.csv"))

pivot = attraction.pivot(index="tier", columns="model", values="attraction_rate").loc[TIER_ORDER]
print(pivot.round(2).to_string())
print()
trend = monotonicity[monotonicity.measure == "distractor_attraction"]
print(f"attraction decreases near->far for "
      f"{int(trend.monotonic_as_expected.sum())}/{len(trend)} models")
print(f"best p-value across models: {trend.p_value.min():.3f} "
      f"(3 ordered bins cannot reach significance)")

### Assemble Figure 4

In [ ]:
from matplotlib.transforms import Bbox

# Two rows rather than one 1x4 strip: in the strip the two confusion matrices were squeezed
# into the middle two of four columns, which left them far too small to read. Row 1 carries
# a and c; row 2 gives both heatmaps a half-width cell each so the cells and their numbers
# can grow.
fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM, 165))
outer = fig.add_gridspec(2, 1, height_ratios=[1, 1.15], hspace=0.62)
top = outer[0].subgridspec(1, 2, width_ratios=[1, 1.25], wspace=0.75)
bottom = outer[1].subgridspec(1, 2, wspace=0.30)

# --- a: tier distribution ---
ax = fig.add_subplot(top[0, 0])
sources = ["OE_native", "MCQ_derived"]
bottom_stack = np.zeros(len(sources))
for tier in TIER_ORDER:
    values = np.array([100 * distribution[s][tier] / distribution[s]["total"]
                       for s in sources])
    ax.bar(np.arange(len(sources)), values, 0.55, bottom=bottom_stack,
           color=TIER_COLORS[tier], label=TIER_SHORT[tier],
           edgecolor="white", linewidth=0.4)
    for x, (value, base) in enumerate(zip(values, bottom_stack)):
        if value > 6:
            ax.text(x, base + value / 2, f"{value:.0f}", ha="center", va="center",
                    fontsize=7, color="white" if tier == "near-miss" else "black")
    bottom_stack += values
ax.set_xticks(np.arange(len(sources)))
ax.set_xticklabels([f"OE-native\n(n={distribution['OE_native']['total']})",
                    f"MCQ-derived\n(n={distribution['MCQ_derived']['total']})"],
                   fontsize=7)
ax.set_ylabel("Wrong answers (%)")
ax.set_ylim(0, 100)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=6.5,
          frameon=False, columnspacing=0.6)
ns.panel_label_below(ax, "a")

# --- c: distractor attraction ---
ax_c = fig.add_subplot(top[0, 1])
x = np.arange(len(TIER_ORDER))
for model in ns.MODEL_ORDER:
    if model not in pivot.columns:
        continue
    ax_c.plot(x, pivot[model].values, marker="o", ms=3.2, lw=0.9,
              color=ns.MODEL_COLORS[model], label=ns.MODEL_LABELS[model])
ax_c.set_xticks(x)
ax_c.set_xticklabels([TIER_SHORT[t] for t in TIER_ORDER], fontsize=7)
ax_c.set_xlabel("Distractor tier", fontsize=7.5)
ax_c.set_ylabel("Distractors selected (%)", fontsize=7.5)
n_consistent = int(trend.monotonic_as_expected.sum())
# The caveat belongs to panel c specifically - three ordered bins cannot reach
# significance, and a reader should not read one off a downward-sloping line. Anchored
# inside c's own axes so it cannot read as floating between panels.
ax_c.set_ylim(top=ax_c.get_ylim()[1] + 4.5)
ax_c.text(0.03, 0.97,
          f"{n_consistent}/{len(trend)} models decrease near\u2192far\n"
          f"(3 bins; all $P$ > 0.3, n.s.)",
          transform=ax_c.transAxes, ha="left", va="top", fontsize=6.5,
          color="#666666", linespacing=1.25)
ns.legend_outside(ax_c, "right", fontsize=6.5)
ns.panel_label_below(ax_c, "c")

# --- b: two confusion matrices, one per judge, now a half-width cell each ---
# Abbreviated row labels so the axis title is not pushed out by the width of "Moderate";
# the full tier names appear on the x axis of the same matrix.
Y_TICK_SHORT = {"near-miss": "Near", "moderate-miss": "Mod.", "far-miss": "Far"}
for offset, (judge_key, matrix) in enumerate(matrices.items()):
    ax = fig.add_subplot(bottom[0, offset])
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    ax.imshow(normalised, cmap="Blues", vmin=0, vmax=0.7, aspect="equal")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{matrix[i, j]}", ha="center", va="center", fontsize=9,
                    color="white" if normalised[i, j] > 0.4 else "black")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels([TIER_SHORT[t] for t in TIER_ORDER], fontsize=8.5,
                       rotation=0, ha="center")
    # Abbreviated so the axis title is not pushed out by "Moderate"; the full tier
    # names are on the x axis of the same matrix.
    ax.set_yticklabels([Y_TICK_SHORT[t] for t in TIER_ORDER] if offset == 0 else [],
                       fontsize=8.5)
    ax.set_xlabel("Judge tier", fontsize=9, labelpad=4)
    if offset == 0:
        ax.set_ylabel("Pathologist tier", fontsize=9)
        b_ylabel_axes = ax
    mean_kappa = tiers[tiers.judge == judge_key].weighted_kappa.mean()
    ax.set_title(f"{'InternVL' if judge_key == 'internvl' else 'Qwen'} judge   "
                 f"$\\kappa$ = {mean_kappa:.2f}", fontsize=8, pad=4)
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Both matrices are panel b; label once, centred under the pair.
    if offset == 0:
        b_left = ax
    else:
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        pair = Bbox.union([b_left.get_window_extent(renderer),
                           ax.get_window_extent(renderer)])
        lowest = min(t.get_window_extent(renderer).y0
                     for a in (b_left, ax) for t in a.get_xticklabels() if t.get_text())
        lowest = min(lowest, b_left.xaxis.label.get_window_extent(renderer).y0)
        centre = fig.transFigure.inverted().transform(
            ((pair.x0 + pair.x1) / 2, lowest - 10 * fig.dpi / 72.0))
        fig.text(centre[0], centre[1], "b", fontsize=ns.fs(10), fontweight="bold",
                 ha="center", va="top")

# Sit the y-axis title just clear of its tick labels. Left to matplotlib it is offset by
# the widest tick ("Moderate"), stranding it ~190 px from a plot only 613 px wide.
# -0.09 axes-fractions places it a few px left of the ticks; measured, not guessed.
fig.canvas.draw()
_renderer = fig.canvas.get_renderer()
_axes_box = b_ylabel_axes.get_window_extent(_renderer)
_tick_left = min(t.get_window_extent(_renderer).x0
                 for t in b_ylabel_axes.get_yticklabels() if t.get_text())
_label_width = b_ylabel_axes.yaxis.label.get_window_extent(_renderer).width
# Right edge of the (rotated) title lands 5 pt left of the leftmost tick glyph.
b_ylabel_axes.yaxis.set_label_coords(
    (_tick_left - 5 * fig.dpi / 72.0 - _label_width / 2 - _axes_box.x0) / _axes_box.width,
    0.5)

paths = ns.save(fig, "fig02_pillar2_wrong_answers")
print("wrote:", paths)
plt.show()

## Draft caption

**Fig. 2 | PathOPEN's wrong answers form a graded difficulty structure, weakly recovered
by VLM judges and directionally supported by model behaviour.**
**a**, Distribution of pathologist-assigned difficulty tiers for wrong answers, split by
source. Tiers are the Benchmark 2 Error Proximity score binned as 2 → near-miss,
1 → moderate-miss, 0 → far-miss; items scored −1 have no tier and are excluded. All three
tiers are populated in both sources, though MCQ-derived distractors skew further toward
far-miss than OE-native ones.
**b**, Tier-assignment agreement between each VLM judge and the pathologist who rated
each item, one confusion matrix per judge. Cells give counts; rows are the pathologist's
tier and columns the judge's. Mass below the diagonal indicates the judge rated
distractors as *harder* (closer to near-miss) than the pathologist did — 57.9% of
InternVL's and 41.0% of Qwen's assignments fall there, against 32.7% and 41.2% on the
diagonal. Quadratic-weighted Cohen's κ, shown above each matrix, is the mean across
Benchmark 2 criteria and wrong-answer slots.
**c**, Behavioural validation. Attraction rate is the percentage of offered distractors
of each tier that a model selected, normalised by tier availability so that a more
frequent tier does not appear more attractive. Five of six models show attraction
decreasing from near-miss to far-miss, the direction predicted if the tiers track genuine
difficulty; with only three ordered bins, Kendall's τ cannot reach significance for any
model (all *P* > 0.3).

**Layout note.** Panels **a** and **c** occupy the top row; panel **b** is the pair of
confusion matrices in the second row, labelled once beneath the pair because both matrices
are the same analysis for different judges. Row abbreviations in **b** are Near / Mod. /
Far; the full tier names appear on the x axis of the same matrix.

---

### Numbers a caption must not get wrong

| quantity | value |
|---|---|
| OE-native tiers (near/mod/far) | 17.2% / 43.0% / 39.8%, n = 668 |
| MCQ-derived tiers | 14.0% / 24.8% / 61.1%, n = 1,773 |
| tier κ range across judges/criteria | 0.02 – 0.25 |
| on-diagonal agreement, InternVL / Qwen | 32.7% / 41.2% |
| judge rates HARDER than pathologist | 57.9% / 41.0% |
| models with predicted attraction direction | 5 of 6 |
| best Kendall *P* | 0.333 |

### What must be said in the text, not the caption

Panel **c** is marked **optional** in §2.2 and is the paper's own nominated "strongest
evidence" for the tiering, because panel **b** is weak. Report it as *directionally
consistent* rather than confirmatory: the sign is right for 5 of 6 models, but three
ordered bins cannot produce a significant Kendall's τ, and the pattern is non-monotonic
for most models (moderate-miss often attracts more than near-miss). Only Patho-R1 shows
a clean gradient on both the attraction and item-accuracy measures.

The judges' systematic over-rating of distractor plausibility in **b** is itself worth a
sentence: it means a VLM judge is not currently a reliable substitute for pathologist
tiering, which bears on how far Benchmark 2 can be automated.
